# 🚗 YOLOv8n — Training Klasifikasi Golongan Kendaraan Tol
**Paper**: Klasifikasi Golongan Kendaraan Tol Lingkar Luar Jakarta Timur  
**Model**: YOLOv8n (Nano) | **GPU**: T4 (Colab Free)  
**Dataset**: 5 kelas GOL I–V | 1.453 gambar

---
## ⚡ Langkah-langkah:
1. Pastikan GPU aktif: **Runtime → Change runtime type → T4 GPU**
2. Jalankan sel dari atas ke bawah secara berurutan
3. Download `best.pt` di sel terakhir

In [ ]:
# ── CELL 1: Cek GPU ─────────────────────────────────────────────────────────
import torch
print('='*50)
print(f'GPU tersedia : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Nama GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print('='*50)

# Kalau GPU: False → Runtime → Change runtime type → T4 GPU → Save

In [ ]:
# ── CELL 2: Install dependencies ─────────────────────────────────────────────
!pip install ultralytics roboflow -q
print('✅ Ultralytics & Roboflow installed')

In [ ]:
# ── CELL 3: Download Dataset dari Roboflow ───────────────────────────────────
# Dataset: Golongan Kendaraan Jalan Tol Lingkar Luar Jakarta Timur
# Sumber : https://universe.roboflow.com/muhammad-rizky-ferdiansyah-00fow

from roboflow import Roboflow

# Pakai API key dari akun Roboflow kamu (gratis)
# Daftar di: https://roboflow.com → Settings → API Keys
API_KEY = "GANTI_DENGAN_API_KEY_KAMU"   # ← ganti ini!

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("muhammad-rizky-ferdiansyah-00fow").project(
    "golongan-kendaraan-jalan-tol-lingkar-luar-jakarta-timur"
)
dataset = project.version(5).download("yolov8")
print(f'✅ Dataset downloaded ke: {dataset.location}')

In [ ]:
# ── CELL 3b (ALTERNATIF): Upload dataset dari komputer lokal ─────────────────
# Jalankan sel ini HANYA jika tidak punya API Key Roboflow
# Zip folder: train/, valid/, test/, data.yaml → upload di sini

# from google.colab import files
# import zipfile, os
# uploaded = files.upload()          # pilih file .zip dataset kamu
# zip_name = list(uploaded.keys())[0]
# with zipfile.ZipFile(zip_name, 'r') as z:
#     z.extractall('/content/dataset')
# print('✅ Dataset extracted ke /content/dataset')

In [ ]:
# ── CELL 4: Buat data.yaml yang benar ────────────────────────────────────────
import os, yaml
from pathlib import Path

# Temukan lokasi dataset
# Jika pakai Roboflow:
DATASET_DIR = Path(dataset.location)   # ganti ke Path('/content/dataset') jika upload manual

data_yaml = {
    'train': str(DATASET_DIR / 'train' / 'images'),
    'val':   str(DATASET_DIR / 'valid' / 'images'),
    'test':  str(DATASET_DIR / 'test'  / 'images'),
    'nc': 5,
    'names': ['GOL I', 'GOL II', 'GOL III', 'GOL IV', 'GOL V']
}

yaml_path = '/content/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print('✅ data.yaml dibuat:')
print(open(yaml_path).read())

# Verifikasi gambar
for split in ['train', 'valid', 'test']:
    p = DATASET_DIR / split / 'images'
    n = len(list(p.glob('*.[jJpP][pPnN][gG]*'))) if p.exists() else 0
    print(f'  {split:6s}: {n} gambar')

In [ ]:
# ── CELL 5: TRAINING YOLOv8n ──────────────────────────────────────────────────
from ultralytics import YOLO
import time

print('='*60)
print('  YOLOv8n — Training Golongan Kendaraan Tol (GPU Mode)')
print('='*60)

model = YOLO('yolov8n.pt')   # download pretrained weights

t0 = time.time()

results = model.train(
    data       = yaml_path,
    epochs     = 100,
    imgsz      = 640,
    batch      = 16,        # GPU T4: batch=16 aman (VRAM ~2GB)
    device     = 0,         # GPU (0 = cuda:0)
    workers    = 2,
    project    = '/content/runs',
    name       = 'vehicle_cls_v1',
    exist_ok   = True,
    patience   = 20,        # early stop jika 20 epoch tidak improve
    save       = True,
    save_period= 10,
    plots      = True,
    verbose    = True,
    # ── Augmentasi (Paper Section 3.3) ──
    hsv_h      = 0.015,
    hsv_s      = 0.7,
    hsv_v      = 0.4,       # brightness/exposure jitter
    degrees    = 0.0,
    translate  = 0.1,
    scale      = 0.5,
    shear      = 0.0,
    perspective= 0.0,
    flipud     = 0.0,
    fliplr     = 0.5,
    mosaic     = 1.0,
    mixup      = 0.0,
    copy_paste = 0.0,
    erasing    = 0.4,       # random erasing ≈ simulasi noise oklusi
)

elapsed = time.time() - t0
h, m = divmod(int(elapsed), 3600)
m, s = divmod(m, 60)
print(f'\n⏱️  Total waktu training: {h}j {m}m {s}d')
print(f'📁 Model disimpan di: {results.save_dir}')

In [ ]:
# ── CELL 6: Evaluasi hasil training ──────────────────────────────────────────
from ultralytics import YOLO

best_pt = f'{results.save_dir}/weights/best.pt'
model   = YOLO(best_pt)
metrics = model.val(data=yaml_path, device=0, verbose=True)

print('\n' + '='*55)
print('  HASIL EVALUASI AKHIR')
print('='*55)
print(f'  mAP@0.50      : {metrics.box.map50:.4f}  ({metrics.box.map50*100:.1f}%)')
print(f'  mAP@0.50:0.95 : {metrics.box.map:.4f}  ({metrics.box.map*100:.1f}%)')
print(f'  Precision     : {metrics.box.mp:.4f}')
print(f'  Recall        : {metrics.box.mr:.4f}')

# Per kelas
CLASS_NAMES = ['GOL I', 'GOL II', 'GOL III', 'GOL IV', 'GOL V']
print(f'\n  {"Kelas":<12} {"AP@50":>10}  {"AP@50-95":>12}')
print(f'  {"-"*36}')
for i, cls in enumerate(CLASS_NAMES):
    a50 = float(metrics.box.ap50[i]) if i < len(metrics.box.ap50) else 0
    a95 = float(metrics.box.ap[i])   if i < len(metrics.box.ap)   else 0
    print(f'  {cls:<12} {a50:>10.4f}  {a95:>12.4f}')
print('='*55)

In [ ]:
# ── CELL 7: Export ke ONNX ────────────────────────────────────────────────────
from ultralytics import YOLO

model = YOLO(best_pt)
onnx_path = model.export(
    format   = 'onnx',
    imgsz    = 640,
    opset    = 12,
    simplify = True,
    device   = 0,
)
print(f'✅ ONNX disimpan: {onnx_path}')

In [ ]:
# ── CELL 8: Download model ke komputer lokal ─────────────────────────────────
import shutil, os
from google.colab import files

# Kemas semua output dalam satu zip
save_dir = str(results.save_dir)
zip_out  = '/content/vehicle_cls_v1_results.zip'
shutil.make_archive('/content/vehicle_cls_v1_results', 'zip', save_dir)

print(f'📦 File yang akan didownload:')
for f in ['best.pt', 'last.pt', 'best.onnx']:
    fp = os.path.join(save_dir, 'weights', f)
    if os.path.exists(fp):
        size_mb = os.path.getsize(fp) / 1e6
        print(f'   ✅ weights/{f} ({size_mb:.1f} MB)')

print(f'\n⬇️  Mendownload zip ({os.path.getsize(zip_out)/1e6:.1f} MB) ...')
files.download(zip_path)

## 📋 Setelah Download

Extract zip, lalu:
1. Salin `weights/best.pt` → `backend/models/best.pt`
2. Salin `weights/best.onnx` → `backend/models/best.onnx`
3. Jalankan backend:
   ```bash
   cd backend
   venv\Scripts\activate
   python main.py
   ```

**Golongan Kendaraan (Standar Jasa Marga):**
| ID | Kelas | Deskripsi |
|----|-------|-----------|
| 0 | GOL I | Sedan / Jip / Pick-up / Bus |
| 1 | GOL II | Truk 2 Gandar |
| 2 | GOL III | Truk 3 Gandar |
| 3 | GOL IV | Truk 4 Gandar |
| 4 | GOL V | Truk 5 Gandar atau lebih |